In [ ]:

import cv2
import numpy as np
import glob
import os
import base64
import json
from ultralytics import YOLO
from google.colab import files
from IPython.display import HTML, display

# --- 1. 배경 이미지 로드 ---
CLEAN_BREADBOARD_IMG = '/content/clean_breadboard.png'
cb_canvas = cv2.imread(CLEAN_BREADBOARD_IMG)

if cb_canvas is None:
    print("🚨 [오류] 'clean_breadboard.png' 파일을 찾을 수 없습니다. 왼쪽 폴더에 업로드 해주세요.")
else:
    CB_HEIGHT, CB_WIDTH, _ = cb_canvas.shape

    # --- 2. 🌟 빵판 구멍 자동 인식 (글씨 및 기호 완벽 차단) ---
    gray = cv2.cvtColor(cb_canvas, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    temp_holes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        bbox_area = w * h
        contour_area = cv2.contourArea(cnt)

        if h == 0 or w == 0 or contour_area == 0: continue

        aspect_ratio = w / float(h)
        extent = contour_area / float(bbox_area)

        if 20 < bbox_area < 1000 and 0.75 < aspect_ratio < 1.3 and extent > 0.65:
            temp_holes.append({"x": x + w//2, "y": y + h//2, "area": bbox_area})

    holes_data = []
    if not temp_holes:
        print("🚨 [오류] 그림 파일에서 구멍을 전혀 찾지 못했습니다!")
    else:
        median_area = np.median([h['area'] for h in temp_holes])

        for h in temp_holes:
            if median_area * 0.7 < h['area'] < median_area * 1.3:
                holes_data.append({"x": h['x'], "y": h['y'], "net_id": None})

        # --- 3. 🌟 4구역(Zone) 기반 Net 그룹화 및 전원선 구분 ---
        x_coords = [h['x'] for h in holes_data]
        min_x, max_x = min(x_coords), max(x_coords)
        span_x = max_x - min_x

        Z1_END = min_x + span_x * 0.12
        Z2_END = min_x + span_x * 0.48
        Z3_START = min_x + span_x * 0.52
        Z4_START = min_x + span_x * 0.88

        TOL = span_x * 0.015

        net_map = {i: i for i in range(len(holes_data))}
        def find(i):
            if net_map[i] == i: return i
            net_map[i] = find(net_map[i])
            return net_map[i]
        def union(i, j):
            root_i, root_j = find(i), find(j)
            if root_i != root_j: net_map[root_i] = root_j

        for i in range(len(holes_data)):
            for j in range(i+1, len(holes_data)):
                h1, h2 = holes_data[i], holes_data[j]
                dx, dy = abs(h1['x'] - h2['x']), abs(h1['y'] - h2['y'])

                if h1['x'] < Z1_END: z1 = 1
                elif h1['x'] < Z2_END: z1 = 2
                elif h1['x'] > Z4_START: z1 = 4
                elif h1['x'] >= Z3_START: z1 = 3
                else: z1 = 0

                if h2['x'] < Z1_END: z2 = 1
                elif h2['x'] < Z2_END: z2 = 2
                elif h2['x'] > Z4_START: z2 = 4
                elif h2['x'] >= Z3_START: z2 = 3
                else: z2 = 0

                if z1 == z2 and z1 != 0:
                    if z1 == 2 or z1 == 3:
                        if dy < TOL: union(i, j)
                    elif z1 == 1 or z1 == 4:
                        if dx < TOL: union(i, j)

        net_groups = {}
        for i in range(len(holes_data)):
            root = find(i)
            if root not in net_groups:
                net_groups[root] = []
            net_groups[root].append(holes_data[i])

        net_dict = {}
        net_counter = 1

        z1_roots, z4_roots = [], []

        for root, holes in net_groups.items():
            avg_x = sum(h['x'] for h in holes) / len(holes)
            if len(holes) < 3:
                net_dict[root] = f"IGNORE_{root}"
                continue

            if avg_x < Z1_END: z1_roots.append((root, avg_x))
            elif avg_x > Z4_START: z4_roots.append((root, avg_x))
            else:
                net_dict[root] = f"NET_{net_counter:03d}"
                net_counter += 1

        z1_roots.sort(key=lambda item: item[1])
        if len(z1_roots) >= 2:
            net_dict[z1_roots[0][0]] = "LEFT_PWR_+"
            net_dict[z1_roots[-1][0]] = "LEFT_PWR_-"

        z4_roots.sort(key=lambda item: item[1])
        if len(z4_roots) >= 2:
            net_dict[z4_roots[0][0]] = "RIGHT_PWR_+"
            net_dict[z4_roots[-1][0]] = "RIGHT_PWR_-"

        valid_nets = set()
        for i in range(len(holes_data)):
            root = find(i)
            net_name = net_dict.get(root, f"NET_UNK_{root}")
            holes_data[i]['net_id'] = net_name
            if "IGNORE" not in net_name:
                valid_nets.add(net_name)

        print(f"🎯 인식된 구멍: {len(holes_data)}개 / 생성된 회로 라인(Net): {len(valid_nets)}개 (성공!)")

        # --- 4. 🌟 모델 추론 (허공이 아닌 빵판 구멍에 완벽 조준 스폰) ---
        weight_paths = glob.glob('/content/runs/detect/*/weights/best.pt')
        if not weight_paths:
            print("🚨 [오류] 학습된 모델을 찾을 수 없습니다.")
        else:
            trained_model = YOLO(max(weight_paths, key=os.path.getctime))

            print("\n👇 [파일 선택] 버튼을 눌러 실제 빵판 사진을 업로드해주세요!")
            uploaded = files.upload()

            if uploaded:
                real_image_path = list(uploaded.keys())[0]
                real_img = cv2.imread(real_image_path)
                real_h, real_w, _ = real_img.shape

                results = trained_model(real_image_path)
                result = results[0]

                # 💡 [핵심 해결 로직] 캔버스 전체가 아니라 '구멍이 모여있는 실제 빵판 영역'의 바운딩 박스를 계산
                min_hx = min([h['x'] for h in holes_data])
                max_hx = max([h['x'] for h in holes_data])
                min_hy = min([h['y'] for h in holes_data])
                max_hy = max([h['y'] for h in holes_data])

                hole_w = max_hx - min_hx
                hole_h = max_hy - min_hy

                # 사진에 찍힌 빵판의 테두리 여백을 고려하여 구멍 영역보다 약 15% 크게 스케일링
                scale = min(hole_w / real_w, hole_h / real_h) * 1.15

                # 구멍 영역의 '정중앙'에 사진의 정중앙이 오도록 오프셋 계산 (허공 스폰 방지!)
                offset_x = min_hx + (hole_w - (real_w * scale)) / 2
                offset_y = min_hy + (hole_h - (real_h * scale)) / 2

                components_data = []
                for i, box in enumerate(result.boxes):
                    class_name = trained_model.names[int(box.cls[0].item())].lower()
                    xmin, ymin, xmax, ymax = map(int, box.xyxy[0].tolist())

                    cb_xmin = int(xmin * scale + offset_x)
                    cb_ymin = int(ymin * scale + offset_y)
                    cb_w = int((xmax - xmin) * scale)
                    cb_h = int((ymax - ymin) * scale)

                    components_data.append({
                        "id": i, "name": f"{class_name}_{i+1}", "type": class_name,
                        "x": cb_xmin, "y": cb_ymin, "w": cb_w, "h": cb_h
                    })

                with open(CLEAN_BREADBOARD_IMG, "rb") as image_file:
                    encoded_string = base64.b64encode(image_file.read()).decode()
                bg_base64 = f"data:image/png;base64,{encoded_string}"

                json_components = json.dumps(components_data)
                json_holes = json.dumps(holes_data)

                # --- 5. HTML+JS 시뮬레이터 (스냅 기능 포함) ---
                html_code = f"""
                <div style="text-align: center; font-family: sans-serif;">
                    <h3 style="color: #333;">💡 부품이 처음부터 빵판 위에 올라왔나요? 자석 버튼으로 마무리하세요!</h3>

                    <div style="background: #eef; padding: 15px; border-radius: 8px; display: inline-block; margin-bottom: 15px; text-align: left; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
                        <strong style="display:block; margin-bottom: 8px; color:#222;">🛠️ 1단계: 미세 조정 (초기 위치가 잘 맞았다면 바로 2단계로!)</strong>
                        <label style="display:inline-block; width: 48%;">↔ 가로 이동: <input type="range" id="shiftX" min="-500" max="500" value="0" oninput="applyGlobalTransform()"></label>
                        <label style="display:inline-block; width: 48%;">↕ 세로 이동: <input type="range" id="shiftY" min="-500" max="500" value="0" oninput="applyGlobalTransform()"></label><br>
                        <label style="display:inline-block; width: 48%;">🔍 전체 크기: <input type="range" id="scaleAll" min="0.3" max="3.0" step="0.05" value="1.0" oninput="applyGlobalTransform()"></label>
                        <label style="display:inline-block; width: 48%;">↔ 가로 폭만: <input type="range" id="scaleX" min="0.5" max="2.0" step="0.05" value="1.0" oninput="applyGlobalTransform()"></label>
                    </div>

                    <div style="display: inline-block; position: relative;">
                        <canvas id="breadboardCanvas" width="{CB_WIDTH}" height="{CB_HEIGHT}"
                                style="border: 2px solid #555; max-width: 100%; height: auto; box-shadow: 2px 2px 10px rgba(0,0,0,0.3);"></canvas>
                    </div>
                    <br>
                    <div style="margin-top: 15px;">
                        <button onclick="snapToGrid()" style="padding: 12px 24px; font-size: 16px; font-weight: bold; cursor: pointer; background-color: #FFA500; color: white; border: none; border-radius: 5px; margin-right: 10px;">
                            🧲 2단계: 구멍에 착! 붙이기 (스냅)
                        </button>
                        <button onclick="generateNetlist()" style="padding: 12px 24px; font-size: 16px; font-weight: bold; cursor: pointer; background-color: #007BFF; color: white; border: none; border-radius: 5px;">
                            ✅ 3단계: 넷리스트 추출
                        </button>
                    </div>
                    <pre id="outputConsole" style="text-align: left; background: #222; color: #0f0; padding: 15px; margin-top: 15px; max-height: 400px; overflow-y: auto; border-radius: 5px; font-size: 15px; width: 80%; margin-left: auto; margin-right: auto; line-height: 1.6;"></pre>
                </div>

                <script>
                    const canvas = document.getElementById('breadboardCanvas');
                    const ctx = canvas.getContext('2d');
                    const outputConsole = document.getElementById('outputConsole');

                    let items = {json_components};
                    let originalItems = JSON.parse(JSON.stringify(items));

                    let holes = {json_holes};
                    let bgImg = new Image();
                    bgImg.src = "{bg_base64}";

                    let isDragging = false, isResizing = false;
                    let dragTarget = null, offsetX, offsetY;
                    const HANDLE_SIZE = 12;

                    let netGroups = {{}};
                    holes.forEach(h => {{
                        if (!netGroups[h.net_id]) netGroups[h.net_id] = [];
                        netGroups[h.net_id].push(h);
                    }});

                    bgImg.onload = () => draw();

                    function applyGlobalTransform() {{
                        let sx = parseInt(document.getElementById('shiftX').value);
                        let sy = parseInt(document.getElementById('shiftY').value);
                        let scale = parseFloat(document.getElementById('scaleAll').value);
                        let scaleX = parseFloat(document.getElementById('scaleX').value);

                        let cx = canvas.width / 2;
                        let cy = canvas.height / 2;

                        for (let i = 0; i < items.length; i++) {{
                            items[i].w = originalItems[i].w * scale * scaleX;
                            items[i].h = originalItems[i].h * scale;
                            items[i].x = (originalItems[i].x - cx) * scale * scaleX + cx + sx;
                            items[i].y = (originalItems[i].y - cy) * scale + cy + sy;
                        }}
                        draw();
                    }}

                    function getNearestHole(px, py) {{
                        let minDist = Infinity, nearest = null;
                        holes.forEach(h => {{
                            let d = Math.hypot(h.x - px, h.y - py);
                            if (d < minDist) {{ minDist = d; nearest = h; }}
                        }});
                        return nearest;
                    }}

                    function snapToGrid() {{
                        items.forEach(item => {{
                            let pins = getPinCoords(item);
                            if (pins.length > 0) {{
                                let nearest = getNearestHole(pins[0].x, pins[0].y);
                                if (nearest) {{
                                    let dx = nearest.x - pins[0].x;
                                    let dy = nearest.y - pins[0].y;
                                    item.x += dx;
                                    item.y += dy;
                                }}
                            }}
                        }});

                        originalItems = JSON.parse(JSON.stringify(items));

                        document.getElementById('shiftX').value = 0;
                        document.getElementById('shiftY').value = 0;
                        document.getElementById('scaleAll').value = 1.0;
                        document.getElementById('scaleX').value = 1.0;

                        draw();
                    }}

                    function draw() {{
                        ctx.clearRect(0, 0, canvas.width, canvas.height);
                        ctx.drawImage(bgImg, 0, 0, canvas.width, canvas.height);

                        ctx.lineWidth = 6;
                        ctx.lineCap = "round";
                        ctx.lineJoin = "round";

                        for (let net in netGroups) {{
                            if (net.includes("IGNORE")) continue;

                            let hList = netGroups[net];
                            if (hList.length > 1) {{
                                let dx = Math.abs(hList[0].x - hList[hList.length-1].x);
                                let dy = Math.abs(hList[0].y - hList[hList.length-1].y);

                                if (dx > dy) hList.sort((a, b) => a.x - b.x);
                                else hList.sort((a, b) => a.y - b.y);

                                if (net.includes("+")) {{
                                    ctx.strokeStyle = "rgba(255, 50, 50, 0.6)";
                                }} else if (net.includes("-")) {{
                                    ctx.strokeStyle = "rgba(50, 50, 255, 0.6)";
                                }} else {{
                                    ctx.strokeStyle = "rgba(0, 220, 50, 0.6)";
                                }}

                                ctx.beginPath();
                                ctx.moveTo(hList[0].x, hList[0].y);
                                for(let i=1; i<hList.length; i++) {{
                                    ctx.lineTo(hList[i].x, hList[i].y);
                                }}
                                ctx.stroke();
                            }}
                        }}

                        items.forEach(item => {{
                            ctx.fillStyle = "rgba(255, 0, 255, 0.15)";
                            ctx.strokeStyle = "rgba(255, 0, 255, 1)";
                            ctx.lineWidth = 2;
                            ctx.fillRect(item.x, item.y, item.w, item.h);
                            ctx.strokeRect(item.x, item.y, item.w, item.h);

                            ctx.fillStyle = "#8B008B";
                            ctx.font = "bold 16px Arial";
                            ctx.fillText(item.name, item.x, item.y - 8);

                            ctx.fillStyle = "red";
                            ctx.fillRect(item.x + item.w - HANDLE_SIZE, item.y + item.h - HANDLE_SIZE, HANDLE_SIZE, HANDLE_SIZE);

                            drawPins(item);
                        }});
                    }}

                    function drawPins(item) {{
                        let pins = getPinCoords(item);
                        pins.forEach(pin => {{
                            let nearest = getNearestHole(pin.x, pin.y);
                            if (nearest) {{
                                ctx.beginPath();
                                ctx.moveTo(pin.x, pin.y); ctx.lineTo(nearest.x, nearest.y);
                                ctx.strokeStyle = "rgba(0,0,0,0.5)"; ctx.setLineDash([5, 5]); ctx.stroke(); ctx.setLineDash([]);

                                ctx.beginPath();
                                ctx.arc(nearest.x, nearest.y, 7, 0, 2 * Math.PI, false);
                                ctx.strokeStyle = "black"; ctx.lineWidth = 2; ctx.stroke();
                            }}

                            ctx.beginPath();
                            ctx.arc(pin.x, pin.y, 6, 0, 2 * Math.PI, false);
                            ctx.fillStyle = pin.color; ctx.fill(); ctx.strokeStyle = 'white'; ctx.stroke();
                        }});
                    }}

                    function getPinCoords(item) {{
                        let pins = [];
                        if (['fet', 'mosfet'].includes(item.type)) {{
                            let mid_x = item.x + (item.w / 2);
                            pins.push({{name: "Source", x: mid_x, y: item.y, color: "blue"}});
                            pins.push({{name: "Drain", x: mid_x, y: item.y + (item.h / 2), color: "green"}});
                            pins.push({{name: "Gate", x: mid_x, y: item.y + item.h, color: "red"}});
                        }} else if (['res', 'led', 'cap', 'jump'].includes(item.type)) {{
                            if (item.w > item.h) {{
                                let mid_y = item.y + (item.h / 2);
                                pins.push({{name: "Pin1", x: item.x, y: mid_y, color: "red"}});
                                pins.push({{name: "Pin2", x: item.x + item.w, y: mid_y, color: "red"}});
                            }} else {{
                                let mid_x = item.x + (item.w / 2);
                                pins.push({{name: "Pin1", x: mid_x, y: item.y, color: "red"}});
                                pins.push({{name: "Pin2", x: mid_x, y: item.y + item.h, color: "red"}});
                            }}
                        }} else {{
                            pins.push({{name: "Pin1", x: item.x, y: item.y + item.h, color: "red"}});
                            pins.push({{name: "Pin2", x: item.x + item.w, y: item.y + item.h, color: "red"}});
                        }}
                        return pins;
                    }}

                    function getMousePos(evt) {{
                        const rect = canvas.getBoundingClientRect();
                        const scaleX = canvas.width / rect.width; const scaleY = canvas.height / rect.height;
                        return {{ x: (evt.clientX - rect.left) * scaleX, y: (evt.clientY - rect.top) * scaleY }};
                    }}

                    canvas.addEventListener('mousedown', function(e) {{
                        const pos = getMousePos(e);
                        for (let i = items.length - 1; i >= 0; i--) {{
                            let item = items[i];
                            if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w && pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{
                                isResizing = true; dragTarget = item; canvas.style.cursor = 'nwse-resize'; return;
                            }}
                            if (pos.x >= item.x && pos.x <= item.x + item.w && pos.y >= item.y && pos.y <= item.y + item.h) {{
                                isDragging = true; dragTarget = item; offsetX = pos.x - item.x; offsetY = pos.y - item.y; canvas.style.cursor = 'grabbing'; return;
                            }}
                        }}
                    }});

                    canvas.addEventListener('mousemove', function(e) {{
                        const pos = getMousePos(e);
                        if (isDragging && dragTarget) {{
                            dragTarget.x = pos.x - offsetX; dragTarget.y = pos.y - offsetY;
                            draw();
                        }}
                        else if (isResizing && dragTarget) {{
                            let newW = pos.x - dragTarget.x; let newH = pos.y - dragTarget.y;
                            if (newW > 20) dragTarget.w = newW; if (newH > 20) dragTarget.h = newH;
                            draw();
                        }}
                    }});

                    canvas.addEventListener('mouseup', function(e) {{
                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default';
                        originalItems = JSON.parse(JSON.stringify(items));
                    }});

                    canvas.addEventListener('mouseleave', function(e) {{
                        isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'default';
                    }});

                    function generateNetlist() {{
                        let netlistMap = {{}};

                        items.forEach(item => {{
                            let pins = getPinCoords(item);
                            pins.forEach(pin => {{
                                let nearestHole = getNearestHole(pin.x, pin.y);
                                if (nearestHole && nearestHole.net_id) {{
                                    if (!netlistMap[nearestHole.net_id]) netlistMap[nearestHole.net_id] = [];
                                    netlistMap[nearestHole.net_id].push(`${{item.name}}(${{pin.name}})`);
                                }}
                            }});
                        }});

                        let out = "🔌 [최종 추출된 전기적 회로 넷리스트]\\n\\n";
                        let validConnections = false;

                        out += "🔗 [상호 연결된 노드]\\n";
                        for (let net in netlistMap) {{
                            if (netlistMap[net].length > 1) {{
                                out += `  - ${{net}} :  ${{netlistMap[net].join("  ↔  ")}}\\n`;
                                validConnections = true;
                            }}
                        }}

                        if (!validConnections) out += "  (아직 연결된 부품이 없습니다. 색칠된 선을 공유하도록 다리를 놔주세요!)\\n";

                        out += "\\n📌 [단일 연결 (혼자 꽂힌 핀)]\\n";
                        for (let net in netlistMap) {{
                            if (netlistMap[net].length === 1) {{
                                out += `  - ${{net}} :  ${{netlistMap[net][0]}}\\n`;
                            }}
                        }}

                        outputConsole.innerText = out;
                    }}
                </script>
                """

                display(HTML(html_code))

# 🌟 1. 필수 시스템 패키지 및 파이썬 라이브러리 설치 (가장 중요!)
!apt-get update -qq
!apt-get install graphviz -y -qq
!pip install schemdraw networkx pydot -q

import cv2
import numpy as np
import glob
import os
import base64
import json
import re
import networkx as nx
from networkx.drawing.nx_pydot import graphviz_layout
import schemdraw
import schemdraw.elements as elm
from ultralytics import YOLO
from google.colab import files, output
from IPython.display import HTML, display, Image

# ==========================================
# 🌟 파트 0: 회로도 자동 생성 함수 (에러 수정 완료)
# ==========================================
def auto_draw_schematic(netlist_text):
    print("\n" + "="*50)
    print("🎨 표준 전자회로 기호로 회로도를 생성합니다... (이 아래에 나타납니다!)")

    try:
        lines = netlist_text.strip().split('\n')
        comp_pins = {}

        for line in lines:
            line = line.strip()
            if (line.startswith('-') or line.startswith('⚡')) and ':' in line:
                parts = line[1:].split(':')
                net_name = parts[0].strip()
                comps_str = parts[1].strip()

                if '+' in net_name: display_net = 'VCC'
                elif '-' in net_name: display_net = 'GND'
                else: display_net = net_name

                comps = comps_str.split('↔')
                for comp in comps:
                    comp = comp.strip()
                    match = re.match(r'(.+)\((.+)\)', comp)
                    if match:
                        cname, pin = match.group(1).strip(), match.group(2).strip()
                        if cname not in comp_pins: comp_pins[cname] = {}
                        comp_pins[cname][pin] = display_net

        G = nx.MultiDiGraph()
        for cname, pins in comp_pins.items():
            keys = list(pins.keys())
            if len(keys) >= 2:
                n1, n2 = pins[keys[0]], pins[keys[1]]
                if n2 == 'VCC' or n1 == 'GND':
                    n1, n2 = n2, n1
                ctype = cname.split('_')[0].lower()
                G.add_edge(n1, n2, type=ctype, name=cname)

        # 시스템 graphviz를 이용해 노드 위치 계산
        try:
            pos = graphviz_layout(G, prog='dot')
        except Exception as e:
            print("⚠️ graphviz_layout 실패, 기본 레이아웃으로 대체합니다.")
            pos = nx.spring_layout(G)
            for k in pos: pos[k] = (pos[k][0]*100, pos[k][1]*100)

        for k in pos:
            pos[k] = (pos[k][0]*0.05, pos[k][1]*0.05)

        file_path = '/content/schematic.png'
        with schemdraw.Drawing(show=False) as d:
            d.config(fontsize=12, lw=2)

            for node, p in pos.items():
                if node == 'VCC': d += elm.Vdd().at(p).label('VCC', loc='top')
                elif node == 'GND': d += elm.Ground().at(p)
                else: d += elm.Dot().at(p)

            for u, v, data in G.edges(data=True):
                ctype = data['type']
                cname = data['name']

                # 🌟 [오류 수정됨] ResistorBox 대신 범용적인 Resistor 또는 IEEE 표준 박스 사용
                if 'res' in ctype: e = elm.Resistor()
                elif 'led' in ctype: e = elm.LED()
                elif 'cap' in ctype: e = elm.Capacitor()
                elif 'jump' in ctype: e = elm.Line()
                else:
                    try:
                        e = elm.ResistorIEEE() # 최근 버전의 네모 박스 기호
                    except:
                        e = elm.Resistor() # 이마저도 없으면 지그재그 저항으로 안전하게 대체

                d += e.at(pos[u]).to(pos[v]).label(cname)

        d.save(file_path)
        display(Image(file_path))
        print("✅ 깔끔한 1자 연결 및 표준 기호 회로도 생성 완료!")

    except Exception as err:
        print(f"🚨 회로도 생성 중 치명적 오류 발생: {err}")
        print("넷리스트가 비어있거나 올바르지 않을 수 있습니다.")

    print("="*50 + "\n")

output.register_callback('auto_draw_schematic', auto_draw_schematic)

# ==========================================
# 🌟 파트 1 ~ 4: 이미지 처리 및 구멍 인식
# ==========================================
CLEAN_BREADBOARD_IMG = '/content/clean_breadboard.png'
cb_canvas = cv2.imread(CLEAN_BREADBOARD_IMG)

if cb_canvas is None:
    print("🚨 [오류] 'clean_breadboard.png' 파일이 필요합니다.")
else:
    CB_HEIGHT, CB_WIDTH, _ = cb_canvas.shape
    gray = cv2.cvtColor(cb_canvas, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    temp_holes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        bbox_area, contour_area = w * h, cv2.contourArea(cnt)
        if h == 0 or w == 0 or contour_area == 0: continue
        aspect, extent = w/float(h), contour_area/float(bbox_area)
        if 20 < bbox_area < 1000 and 0.75 < aspect < 1.3 and extent > 0.65:
            temp_holes.append({"x": x + w//2, "y": y + h//2, "area": bbox_area})

    holes_data = []
    if not temp_holes:
        print("🚨 구멍 인식 실패")
    else:
        median_area = np.median([h['area'] for h in temp_holes])
        for h in temp_holes:
            if median_area * 0.7 < h['area'] < median_area * 1.3:
                holes_data.append({"x": h['x'], "y": h['y'], "net_id": None})

        x_coords = [h['x'] for h in holes_data]
        min_x, max_x = min(x_coords), max(x_coords)
        span_x = max_x - min_x
        Z1_END = min_x + span_x * 0.12
        Z2_END = min_x + span_x * 0.48
        Z3_START = min_x + span_x * 0.52
        Z4_START = min_x + span_x * 0.88

        net_map = {i: i for i in range(len(holes_data))}
        def find(i):
            if net_map[i] == i: return i
            net_map[i] = find(net_map[i]); return net_map[i]
        def union(i, j):
            ri, rj = find(i), find(j)
            if ri != rj: net_map[ri] = rj

        TOL = span_x * 0.015
        for i in range(len(holes_data)):
            for j in range(i+1, len(holes_data)):
                h1, h2 = holes_data[i], holes_data[j]
                dx, dy = abs(h1['x'] - h2['x']), abs(h1['y'] - h2['y'])

                if h1['x'] < Z1_END: z1 = 1
                elif h1['x'] < Z2_END: z1 = 2
                elif h1['x'] > Z4_START: z1 = 4
                elif h1['x'] >= Z3_START: z1 = 3
                else: z1 = 0

                if h2['x'] < Z1_END: z2 = 1
                elif h2['x'] < Z2_END: z2 = 2
                elif h2['x'] > Z4_START: z2 = 4
                elif h2['x'] >= Z3_START: z2 = 3
                else: z2 = 0

                if z1 == z2 and z1 != 0:
                    if z1 == 2 or z1 == 3:
                        if dy < TOL: union(i, j)
                    elif z1 == 1 or z1 == 4:
                        if dx < TOL: union(i, j)

        net_groups = {}
        for i in range(len(holes_data)):
            root = find(i)
            if root not in net_groups: net_groups[root] = []
            net_groups[root].append(holes_data[i])

        net_dict = {}
        z1_roots, z4_roots = [], []
        net_counter = 1
        for root, hlist in net_groups.items():
            ax = sum(h['x'] for h in hlist) / len(hlist)
            if len(hlist) < 3: net_dict[root] = "IGNORE"; continue
            if ax < Z1_END: z1_roots.append((root, ax))
            elif ax > Z4_START: z4_roots.append((root, ax))
            else: net_dict[root] = f"NET_{net_counter:03d}"; net_counter += 1

        z1_roots.sort(key=lambda x: x[1])
        if len(z1_roots) >= 2: net_dict[z1_roots[0][0]], net_dict[z1_roots[-1][0]] = "LEFT_PWR_+", "LEFT_PWR_-"
        z4_roots.sort(key=lambda x: x[1])
        if len(z4_roots) >= 2: net_dict[z4_roots[0][0]], net_dict[z4_roots[-1][0]] = "RIGHT_PWR_+", "RIGHT_PWR_-"

        for i in range(len(holes_data)):
            holes_data[i]['net_id'] = net_dict.get(find(i), "UNKNOWN")

        weight_paths = glob.glob('/content/runs/detect/*/weights/best.pt')
        if not weight_paths:
            print("🚨 [오류] 모델 가중치 파일이 없습니다.")
        else:
            trained_model = YOLO(max(weight_paths, key=os.path.getctime))

            print("\n👇 실제 빵판 사진을 업로드해주세요!")
            uploaded = files.upload()

            if uploaded:
                real_image_path = list(uploaded.keys())[0]
                real_img = cv2.imread(real_image_path)
                results = trained_model(real_image_path)[0]

                min_hx, max_hx = min([h['x'] for h in holes_data]), max([h['x'] for h in holes_data])
                min_hy, max_hy = min([h['y'] for h in holes_data]), max([h['y'] for h in holes_data])
                hw, hh = max_hx - min_hx, max_hy - min_hy
                sc = min(hw / real_img.shape[1], hh / real_img.shape[0]) * 1.15
                ox, oy = min_hx + (hw - real_img.shape[1]*sc)/2, min_hy + (hh - real_img.shape[0]*sc)/2

                components_data = []
                for i, box in enumerate(results.boxes):
                    cname = trained_model.names[int(box.cls[0].item())].lower()
                    b = box.xyxy[0].tolist()
                    components_data.append({"id": i, "name": f"{cname}_{i+1}", "type": cname, "x": int(b[0]*sc+ox), "y": int(b[1]*sc+oy), "w": int((b[2]-b[0])*sc), "h": int((b[3]-b[1])*sc)})

                with open(CLEAN_BREADBOARD_IMG, "rb") as f:
                    bg_base64 = f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

                # ==========================================
                # 🌟 파트 5: HTML 시뮬레이터 (UI 완벽 적용)
                # ==========================================
                html_code = f"""
                <div style="text-align: center; font-family: sans-serif; background: #f9f9f9; padding: 20px; border-radius: 15px;">
                    <div style="display: flex; justify-content: center; gap: 20px; flex-wrap: wrap; margin-bottom: 15px;">

                        <div style="background: #eef; padding: 15px; border-radius: 8px; text-align: left; box-shadow: 2px 2px 5px rgba(0,0,0,0.1); width: 350px;">
                            <strong style="display:block; margin-bottom: 8px; color:#222;">🛠️ 1단계: 전체 일괄 조절</strong>
                            <label style="display:inline-block; width: 48%;">↔ 가로: <input type="range" id="shiftX" min="-500" max="500" value="0" oninput="applyGlobalTransform()"></label>
                            <label style="display:inline-block; width: 48%;">↕ 세로: <input type="range" id="shiftY" min="-500" max="500" value="0" oninput="applyGlobalTransform()"></label>
                            <label style="display:inline-block; width: 48%;">🔍 크기: <input type="range" id="scaleAll" min="0.3" max="3.0" step="0.05" value="1.0" oninput="applyGlobalTransform()"></label>
                            <label style="display:inline-block; width: 48%;">↔ 폭만: <input type="range" id="scaleX" min="0.5" max="2.0" step="0.05" value="1.0" oninput="applyGlobalTransform()"></label>
                        </div>

                        <div style="background: #e8f5e9; padding: 15px; border-radius: 8px; text-align: left; box-shadow: 2px 2px 5px rgba(0,0,0,0.1); width: 350px;">
                            <strong style="display:block; margin-bottom: 8px; color:#222;">🧰 2단계: 개별 부품 추가/삭제</strong>
                            <select id="newCompType" style="padding: 6px; font-size: 14px; border-radius: 4px; border: 1px solid #ccc; margin-right: 5px;">
                                <option value="res">저항 (Res)</option>
                                <option value="led">LED</option>
                                <option value="cap">커패시터 (Cap)</option>
                                <option value="jump">점퍼선 (Jump)</option>
                                <option value="fet">트랜지스터 (FET)</option>
                            </select>
                            <button onclick="addNewComponent()" style="padding: 6px 12px; font-size: 14px; cursor: pointer; background-color: #28a745; color: white; border: none; border-radius: 4px;">➕ 추가</button>
                            <br><br>
                            <button onclick="deleteSelectedItem()" style="padding: 6px 12px; font-size: 14px; cursor: pointer; background-color: #dc3545; color: white; border: none; border-radius: 4px; width: 100%;">
                                🗑️ 선택 삭제 (또는 Delete 키)
                            </button>
                            <span style="display:block; margin-top:8px; font-size:12px; color:#666;">💡 소자 우측 하단 빨간 점을 드래그해 크기를 조절하세요.</span>
                        </div>
                    </div>

                    <div style="display: inline-block; position: relative;">
                        <canvas id="breadboardCanvas" width="{CB_WIDTH}" height="{CB_HEIGHT}" style="border: 3px solid #333; border-radius:10px; max-width: 95%; height: auto; outline:none; cursor: crosshair;" tabindex="0"></canvas>
                    </div>

                    <div style="margin-top: 20px; display: flex; justify-content: center; gap: 15px;">
                        <button onclick="snapToGrid()" style="padding: 15px 30px; font-size: 18px; font-weight: bold; background: #FFA500; color: white; border: none; border-radius: 8px; cursor: pointer;">🧲 3. 구멍에 스냅</button>
                        <button onclick="outputNetlist()" style="padding: 15px 30px; font-size: 18px; font-weight: bold; background: #17a2b8; color: white; border: none; border-radius: 8px; cursor: pointer;">📜 4. 넷리스트 출력</button>
                        <button onclick="drawSchematic()" style="padding: 15px 30px; font-size: 18px; font-weight: bold; background: #6f42c1; color: white; border: none; border-radius: 8px; cursor: pointer;">🎨 5. 회로도 보기</button>
                    </div>

                    <pre id="outputConsole" style="text-align: left; background: #1e1e1e; color: #adff2f; padding: 20px; margin-top: 20px; border-radius: 8px; font-size: 14px; width: 90%; margin-left: auto; margin-right: auto; overflow-x: auto; line-height: 1.6;"></pre>
                </div>

                <script>
                    const canvas = document.getElementById('breadboardCanvas');
                    const ctx = canvas.getContext('2d');
                    const outputConsole = document.getElementById('outputConsole');

                    let items = {json.dumps(components_data)};
                    let originalItems = JSON.parse(JSON.stringify(items));
                    let holes = {json.dumps(holes_data)};
                    let bgImg = new Image();
                    bgImg.src = "{bg_base64}";

                    let isDragging = false, isResizing = false;
                    let dragTarget = null, offsetX, offsetY;
                    let selectedItem = null;
                    const HANDLE_SIZE = 12;

                    let netGroups = {{}};
                    holes.forEach(h => {{
                        if (!netGroups[h.net_id]) netGroups[h.net_id] = [];
                        netGroups[h.net_id].push(h);
                    }});

                    bgImg.onload = () => draw();

                    function addNewComponent() {{
                        const type = document.getElementById('newCompType').value;
                        let w = 60, h = 20;
                        if (['fet', 'mosfet', 'bjt'].includes(type)) {{ w = 40; h = 60; }}
                        let newItem = {{ id: Date.now(), name: type + '_' + Math.floor(Math.random() * 1000), type: type, x: canvas.width / 2 - w / 2, y: canvas.height / 2 - h / 2, w: w, h: h }};
                        items.push(newItem); selectedItem = newItem; originalItems = JSON.parse(JSON.stringify(items)); draw(); canvas.focus();
                    }}

                    function deleteSelectedItem() {{
                        if (selectedItem) {{ items = items.filter(item => item !== selectedItem); selectedItem = null; originalItems = JSON.parse(JSON.stringify(items)); draw(); }}
                    }}

                    window.addEventListener('keydown', function(e) {{
                        if (e.key === 'Delete' || e.key === 'Backspace') {{
                            if(e.target.tagName !== 'INPUT' && e.target.tagName !== 'SELECT') {{ e.preventDefault(); deleteSelectedItem(); }}
                        }}
                    }});

                    function applyGlobalTransform() {{
                        let sx = parseInt(document.getElementById('shiftX').value);
                        let sy = parseInt(document.getElementById('shiftY').value);
                        let scale = parseFloat(document.getElementById('scaleAll').value);
                        let scaleX = parseFloat(document.getElementById('scaleX').value);
                        let cx = canvas.width / 2; let cy = canvas.height / 2;
                        for (let i = 0; i < items.length; i++) {{
                            items[i].w = originalItems[i].w * scale * scaleX; items[i].h = originalItems[i].h * scale;
                            items[i].x = (originalItems[i].x - cx) * scale * scaleX + cx + sx; items[i].y = (originalItems[i].y - cy) * scale + cy + sy;
                        }}
                        draw();
                    }}

                    function getNearestHole(px, py) {{
                        let minDist = Infinity, nearest = null;
                        holes.forEach(h => {{ let d = Math.hypot(h.x - px, h.y - py); if (d < minDist) {{ minDist = d; nearest = h; }} }});
                        return nearest;
                    }}

                    function snapToGrid() {{
                        items.forEach(item => {{
                            let pins = getPinCoords(item);
                            if (pins.length > 0) {{
                                let nearest = getNearestHole(pins[0].x, pins[0].y);
                                if (nearest) {{ item.x += (nearest.x - pins[0].x); item.y += (nearest.y - pins[0].y); }}
                            }}
                        }});
                        originalItems = JSON.parse(JSON.stringify(items));
                        document.getElementById('shiftX').value = 0; document.getElementById('shiftY').value = 0; document.getElementById('scaleAll').value = 1.0; document.getElementById('scaleX').value = 1.0;
                        draw();
                    }}

                    function getNetlist() {{
                        let netMap = {{}};
                        items.forEach(item => {{
                            getPinCoords(item).forEach(pin => {{
                                let hole = getNearestHole(pin.x, pin.y);
                                if(hole && hole.net_id !== "UNKNOWN" && hole.net_id !== "IGNORE") {{
                                    if(!netMap[hole.net_id]) netMap[hole.net_id] = [];
                                    netMap[hole.net_id].push(`${{item.name}}(${{pin.name}})`);
                                }}
                            }});
                        }});
                        let out = "🔌 [전원 및 회로 연결 넷리스트]\\n"; let pwr = "\\n⚡ [전원 레일 연결 상태]\\n"; let nodes = "\\n🔗 [신호 노드 연결 상태]\\n";
                        Object.keys(netMap).forEach(net => {{
                            let line = `  - ${{net}} : ${{netMap[net].join(" ↔ ")}}\\n`;
                            if(net.includes("PWR")) pwr += line; else nodes += line;
                        }});
                        return out + pwr + nodes;
                    }}

                    function outputNetlist() {{ outputConsole.innerText = getNetlist(); }}

                    function drawSchematic() {{
                        const netlist = getNetlist();
                        outputConsole.innerText = "🚀 회로도 데이터를 파이썬으로 전송 중... (아래쪽에 회로도가 생성됩니다)";
                        google.colab.kernel.invokeFunction('auto_draw_schematic', [netlist], {{}});
                    }}

                    function draw() {{
                        ctx.clearRect(0, 0, canvas.width, canvas.height); ctx.drawImage(bgImg, 0, 0, canvas.width, canvas.height);
                        ctx.lineWidth = 6; ctx.lineCap = "round"; ctx.lineJoin = "round";

                        for (let net in netGroups) {{
                            if (net.includes("IGNORE")) continue;
                            let hList = netGroups[net];
                            if (hList.length > 1) {{
                                let dx = Math.abs(hList[0].x - hList[hList.length-1].x); let dy = Math.abs(hList[0].y - hList[hList.length-1].y);
                                if (dx > dy) hList.sort((a, b) => a.x - b.x); else hList.sort((a, b) => a.y - b.y);

                                if (net.includes("+")) ctx.strokeStyle = "rgba(255, 50, 50, 0.6)"; else if (net.includes("-")) ctx.strokeStyle = "rgba(50, 50, 255, 0.6)"; else ctx.strokeStyle = "rgba(0, 220, 50, 0.6)";
                                ctx.beginPath(); ctx.moveTo(hList[0].x, hList[0].y); for(let i=1; i<hList.length; i++) ctx.lineTo(hList[i].x, hList[i].y); ctx.stroke();
                            }}
                        }}

                        items.forEach(item => {{
                            if (item === selectedItem) {{ ctx.shadowColor = "rgba(255, 165, 0, 0.8)"; ctx.shadowBlur = 10; ctx.strokeStyle = "#FF8C00"; ctx.lineWidth = 3;
                            }} else {{ ctx.shadowBlur = 0; ctx.strokeStyle = "rgba(255, 0, 255, 1)"; ctx.lineWidth = 2; }}

                            ctx.fillStyle = "rgba(255, 0, 255, 0.15)"; ctx.fillRect(item.x, item.y, item.w, item.h); ctx.strokeRect(item.x, item.y, item.w, item.h); ctx.shadowBlur = 0;
                            ctx.fillStyle = "#8B008B"; ctx.font = "bold 16px Arial"; ctx.fillText(item.name, item.x, item.y - 8);
                            ctx.fillStyle = "red"; ctx.fillRect(item.x + item.w - HANDLE_SIZE, item.y + item.h - HANDLE_SIZE, HANDLE_SIZE, HANDLE_SIZE);
                            drawPins(item);
                        }});
                    }}

                    function drawPins(item) {{
                        getPinCoords(item).forEach(pin => {{
                            let nearest = getNearestHole(pin.x, pin.y);
                            if (nearest) {{
                                ctx.beginPath(); ctx.moveTo(pin.x, pin.y); ctx.lineTo(nearest.x, nearest.y); ctx.strokeStyle = "rgba(0,0,0,0.5)"; ctx.setLineDash([5, 5]); ctx.stroke(); ctx.setLineDash([]);
                                ctx.beginPath(); ctx.arc(nearest.x, nearest.y, 7, 0, 2 * Math.PI, false); ctx.strokeStyle = "black"; ctx.lineWidth = 2; ctx.stroke();
                            }}
                            ctx.beginPath(); ctx.arc(pin.x, pin.y, 6, 0, 2 * Math.PI, false); ctx.fillStyle = pin.color; ctx.fill(); ctx.strokeStyle = 'white'; ctx.stroke();
                        }});
                    }}

                    function getPinCoords(item) {{
                        let pins = [];
                        if (['fet', 'mosfet', 'bjt'].includes(item.type)) {{
                            let mid_x = item.x + (item.w / 2);
                            pins.push({{name: "Source", x: mid_x, y: item.y, color: "blue"}}); pins.push({{name: "Drain", x: mid_x, y: item.y + (item.h / 2), color: "green"}}); pins.push({{name: "Gate", x: mid_x, y: item.y + item.h, color: "red"}});
                        }} else if (['res', 'led', 'cap', 'jump'].includes(item.type)) {{
                            if (item.w > item.h) {{
                                let mid_y = item.y + (item.h / 2);
                                pins.push({{name: "Pin1", x: item.x, y: mid_y, color: "red"}}); pins.push({{name: "Pin2", x: item.x + item.w, y: mid_y, color: "red"}});
                            }} else {{
                                let mid_x = item.x + (item.w / 2);
                                pins.push({{name: "Pin1", x: mid_x, y: item.y, color: "red"}}); pins.push({{name: "Pin2", x: mid_x, y: item.y + item.h, color: "red"}});
                            }}
                        }} else {{
                            pins.push({{name: "Pin1", x: item.x, y: item.y + item.h, color: "red"}}); pins.push({{name: "Pin2", x: item.x + item.w, y: item.y + item.h, color: "red"}});
                        }}
                        return pins;
                    }}

                    function getMousePos(evt) {{ const rect = canvas.getBoundingClientRect(); return {{ x: (evt.clientX - rect.left) * (canvas.width / rect.width), y: (evt.clientY - rect.top) * (canvas.height / rect.height) }}; }}

                    canvas.addEventListener('mousedown', function(e) {{
                        const pos = getMousePos(e); let clickedOnItem = false; isResizing = false;
                        for (let i = items.length - 1; i >= 0; i--) {{
                            let item = items[i];
                            if (pos.x >= item.x + item.w - HANDLE_SIZE && pos.x <= item.x + item.w && pos.y >= item.y + item.h - HANDLE_SIZE && pos.y <= item.y + item.h) {{
                                isResizing = true; dragTarget = item; selectedItem = item; canvas.style.cursor = 'nwse-resize'; clickedOnItem = true; break;
                            }}
                            if (pos.x >= item.x && pos.x <= item.x + item.w && pos.y >= item.y && pos.y <= item.y + item.h) {{
                                isDragging = true; dragTarget = item; selectedItem = item; offsetX = pos.x - item.x; offsetY = pos.y - item.y; canvas.style.cursor = 'grabbing'; clickedOnItem = true; break;
                            }}
                        }}
                        if (!clickedOnItem) selectedItem = null; draw();
                    }});

                    canvas.addEventListener('mousemove', function(e) {{
                        let px = getMousePos(e).x; let py = getMousePos(e).y;
                        if (isDragging && dragTarget) {{ dragTarget.x = px - offsetX; dragTarget.y = py - offsetY; draw(); }}
                        else if (isResizing && dragTarget) {{ let newW = px - dragTarget.x; let newH = py - dragTarget.y; if (newW > 20) dragTarget.w = newW; if (newH > 10) dragTarget.h = newH; draw(); }}
                    }});

                    canvas.addEventListener('mouseup', function(e) {{ isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'crosshair'; originalItems = JSON.parse(JSON.stringify(items)); }});
                    canvas.addEventListener('mouseleave', function(e) {{ isDragging = false; isResizing = false; dragTarget = null; canvas.style.cursor = 'crosshair'; }});
                </script>
                """
                display(HTML(html_code))